#Silver Layer Validation 
🎯 Goal

Validate the Silver layer to ensure:

- data is clean and reliable
- no critical nulls in key columns
- values are consistent and standardized
- schema is structured and reusable
- ready for Gold layer consumption

In [0]:
import yaml
from pyspark.sql import functions as F

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

print("Catalog:", catalog)
print("Raw Schema:", raw_schema)
print("Refined Schema:", refined_schema)

In [0]:
df_events = spark.table(f"{catalog}.{refined_schema}.silver_grid_events")
df_assets = spark.table(f"{catalog}.{refined_schema}.silver_asset_reference")
df_weather = spark.table(f"{catalog}.{refined_schema}.silver_weather")

In [0]:
print("=== ROW COUNTS ===")

print("Grid Events:", df_events.count())
print("Assets:", df_assets.count())
print("Weather:", df_weather.count())

In [0]:
print("=== NULL CHECK: GRID EVENTS ===")

df_events.select(
    F.count(F.when(F.col("event_id").isNull(), 1)).alias("null_event_id"),
    F.count(F.when(F.col("asset_id").isNull(), 1)).alias("null_asset_id"),
    F.count(F.when(F.col("region").isNull(), 1)).alias("null_region")
).show()

In [0]:
print("=== NULL CHECK: ASSETS ===")

df_assets.select(
    F.count(F.when(F.col("asset_id").isNull(), 1)).alias("null_asset_id"),
    F.count(F.when(F.col("asset_name").isNull(), 1)).alias("null_asset_name"),
    F.count(F.when(F.col("region").isNull(), 1)).alias("null_region")
).show()

In [0]:
print("=== SEVERITY DISTRIBUTION ===")

df_events.groupBy("severity", "severity_band").count().show()

In [0]:
print("=== INVALID DURATION CHECK ===")

df_events.filter(F.col("duration_minutes") < 0).count()

In [0]:
print("=== WEATHER OUTLIERS ===")

df_weather.filter(
    (F.col("temperature_c") < -50) | (F.col("temperature_c") > 60)
).count()

In [0]:
print("=== SCHEMAS ===")

df_events.printSchema()
df_assets.printSchema()
df_weather.printSchema()

In [0]:
print("=== SAMPLE DATA ===")

df_events.show(10, truncate=False)
df_assets.show(10, truncate=False)
df_weather.show(10, truncate=False)

In [0]:
print("=== ORPHAN ASSET IDS IN EVENTS ===")

df_events.select("asset_id").join(
    df_assets.select("asset_id"),
    "asset_id",
    "left_anti"
).show()